In [ ]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
from pathlib import Path
import re
import warnings

In [ ]:
#Creation of initial sepsis and control split from sepsis3 list
sepsis3 = pd.read_csv("sepsis3-df.csv")
sepsis3 = sepsis3.rename(columns={'hadm_id':'HADM_ID'})

neonates = sepsis3[sepsis3['age'] == 0]
neonate_sepsis = neonates[neonates['sepsis-3'] == 1]
neonate_control = neonates[neonates['sepsis-3'] == 0]

control_ids_hadm = neonate_control['HADM_ID']
sepsis_ids_hadm = neonate_sepsis['HADM_ID']

In [ ]:
#Creation of auxillary datasets and subject_id to hadm_id translation
patients = pd.read_csv("/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/PATIENTS.csv")
admissions = pd.read_csv("/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/ADMISSIONS.csv")
mapping_ids = admissions[['SUBJECT_ID', 'HADM_ID']]

sepsis_admissions = admissions[admissions['HADM_ID'].isin(sepsis_ids_hadm)]
control_admissions = admissions[admissions['HADM_ID'].isin(control_ids_hadm)]

sepsis_ids = sepsis_admissions['SUBJECT_ID']
control_ids = control_admissions['SUBJECT_ID']

sepsis_patients = patients[patients['SUBJECT_ID'].isin(sepsis_ids)]
control_patients = patients[patients['SUBJECT_ID'].isin(control_ids)]

In [ ]:
#Collection of physiological data (control), filtered by select vitals
chunk_size = 5000000
chunk_vars = {}
vitals_map = {
    'heart_rate': [211, 220045],
    'respiratory_rate': [3603, 3337, 618, 220210],
    'rr_set':[619],
    'sp02': [646],
    'sa02': [834],
    'fio2': [190, 3420, 2981, 7570],
    'temp_c': [3655, 676, 677, 223762],
    'temp_f': [645, 678, 679, 223761],
    'temp_axillary_f':[3652],
    'temp_rectal_f':[3654, 6643],
    'o2_tcp': [3647, 3651],
    'bp_sys': [51, 3325, 455, 3313],
    'bp_dia': [8502, 8555],
    'map': [3324, 456, 3312, 52]
}

all_vital_ids = [idx for ids in vitals_map.values() for idx in ids]

reader = pd.read_csv(
    "/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/CHARTEVENTS.csv", 
    chunksize=chunk_size,
    usecols=['SUBJECT_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM']
)

last_index = 0
for i, chunk in enumerate(reader):
    var_name = f'df_chunk_{i}'
    chunk = chunk[chunk['SUBJECT_ID'].isin(control_ids)]
    chunk = chunk[chunk['ITEMID'].isin(all_vital_ids)]
    id_to_name = {idx: name for name, ids in vitals_map.items() for idx in ids}
    chunk['LABEL'] = chunk['ITEMID'].map(id_to_name)
    chunk_vars[var_name] = chunk
    last_index = i
    print(f"\rProcessing: {var_name} | Rows in chunk: {len(chunk)}", end="", flush=True)

chunk_list = []
for key in list(chunk_vars.keys()):
    df = chunk_vars[key]
    if df.empty:
        continue
    chunk_list.append(df)

control_phys = pd.concat(chunk_list, ignore_index=True)
control_phys['CHARTTIME'] = pd.to_datetime(control_phys['CHARTTIME'])
control_phys = control_phys.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    
total_rows = sum(len(c) for c in chunk_vars.values())
print(f"\n\n--- Execution Complete ---")
print(f"Total Chunks Created: {last_index + 1}")
print(f"Total Filtered Rows Stored: {total_rows:,}")
control_phys.head()

In [ ]:
#Collection of physiological data (sepsis)
chunk_vars_sepsis = {}

reader = pd.read_csv(
    "/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/CHARTEVENTS.csv", 
    chunksize=chunk_size,
    usecols=['SUBJECT_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM']
)

last_index = 0
for i, chunk in enumerate(reader):
    var_name = f'df_chunk_{i}'
    chunk = chunk[chunk['SUBJECT_ID'].isin(sepsis_ids)]
    chunk = chunk[chunk['ITEMID'].isin(all_vital_ids)]
    chunk['LABEL'] = chunk['ITEMID'].map(id_to_name)
    chunk_vars[var_name] = chunk
    last_index = i
    print(f"\rProcessing: {var_name} | Rows in chunk: {len(chunk)}", end="", flush=True)

chunk_list = []
for key in list(chunk_vars.keys()):
    df = chunk_vars[key]
    if df.empty:
        continue
    chunk_list.append(df)

sepsis_phys = pd.concat(chunk_list, ignore_index=True)
sepsis_phys['CHARTTIME'] = pd.to_datetime(sepsis_phys['CHARTTIME'])
sepsis_phys = sepsis_phys.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    
total_rows = sum(len(c) for c in chunk_vars.values())
print(f"\n\n--- Execution Complete ---")
print(f"Total Chunks Created: {last_index + 1}")
print(f"Total Filtered Rows Stored: {total_rows:,}")
sepsis_phys.head()

In [ ]:
#Conducting density analysis (control)
def perform_density_audit(df):
    sparsity = df.groupby('LABEL').agg(
        total_records=('VALUENUM', 'count'),
        unique_patients=('SUBJECT_ID', 'nunique'),
        mean_val=('VALUENUM', 'mean'),
        std_val=('VALUENUM', 'std')
    )
    sparsity['avg_records_per_patient'] = sparsity['total_records'] / sparsity['unique_patients']
    
    patient_vital_counts = df.pivot_table(
        index='SUBJECT_ID', 
        columns='LABEL', 
        values='VALUENUM', 
        aggfunc='count'
    ).fillna(0)
    
    df = df.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    df['time_diff'] = df.groupby(['SUBJECT_ID', 'LABEL'])['CHARTTIME'].diff().dt.total_seconds() / 60
    
    time_gaps = df.groupby('LABEL')['time_diff'].describe()[['50%', '75%', 'max']]
    time_gaps.columns = ['median_gap_mins', '75th_percentile_gap', 'max_gap']

    stay_durations = df.groupby('SUBJECT_ID')['CHARTTIME'].agg(['min', 'max'])
    stay_durations['total_hours'] = (stay_durations['max'] - stay_durations['min']).dt.total_seconds() / 3600
    
    return sparsity, patient_vital_counts, time_gaps, stay_durations


sparsity_stats, patient_completeness, gap_stats, stay_stats = perform_density_audit(control_phys)

print("\n### VARIABLE SPARSITY ###")
print(sparsity_stats)

print("\n### TYPICAL TIME GAPS (MINUTES) ###")
print(gap_stats)

print(f"\nAverage monitoring duration: {stay_stats['total_hours'].mean():.2f} hours")

In [ ]:
#Conducting density analysis (sepsis)
sparsity_stats, patient_completeness, gap_stats, stay_stats = perform_density_audit(sepsis_phys)

print("\n### VARIABLE SPARSITY ###")
print(sparsity_stats)

print("\n### TYPICAL TIME GAPS (MINUTES) ###")
print(gap_stats)

print(f"\nAverage monitoring duration: {stay_stats['total_hours'].mean():.2f} hours")

In [ ]:
# Loading pre-loaded chartevent data

sepsis_phys = pd.read_csv("sepsis_phys_new.csv")
control_phys = pd.read_csv("control_phys_new.csv")

In [ ]:
# Adding HADM_ID to chartevent data

control_phys = control_phys.merge(mapping_ids[['SUBJECT_ID', 'HADM_ID']], on='SUBJECT_ID', how='left')
sepsis_phys = sepsis_phys.merge(mapping_ids[['SUBJECT_ID', 'HADM_ID']], on='SUBJECT_ID', how='left')

In [ ]:
# Cleans and standardizes neonatal physiological data

def clean_and_standardize_vitals(df, df_admissions=None, mapping_ids=None):
    df = df.copy()
    
    # TEMPERATURE CONVERSION & UNIFICATION
    is_f = df['LABEL'].isin(['temp_axillary_f', 'temp_rectal_f'])
    df.loc[is_f, 'VALUENUM'] = (df.loc[is_f, 'VALUENUM'] - 32) * (5.0 / 9.0)
    
    is_ax = df['LABEL'] == 'temp_axillary_f'
    df.loc[is_ax, 'VALUENUM'] += 0.5
    
    df['UNIFIED_LABEL'] = df['LABEL']
    temp_labels = ['temp_c', 'temp_axillary_f', 'temp_rectal_f']
    df.loc[df['LABEL'].isin(temp_labels), 'UNIFIED_LABEL'] = 'temperature'

    # OUTLIER CLIPPING (Neonatal Ranges)
    bounds = {
        'heart_rate': (30, 250),
        'respiratory_rate': (10, 100),
        'sa02': (40, 100),
        'fio2': (20, 100),
        'map': (20, 100),
        'bp_sys': (30, 200),
        'bp_dia': (15, 170),
        'temp_c': (32, 42),          
        'temp_axillary_f': (32, 42), 
        'temp_rectal_f': (32, 42),
        'o2_tcp': (20, 100) 
    }

    for label, (lower, upper) in bounds.items():
        mask = df['LABEL'] == label
        df.loc[mask, 'VALUENUM'] = df.loc[mask, 'VALUENUM'].clip(lower=lower, upper=upper)

    # BLOOD PRESSURE CALCULATION (MAP)
    bp_df = df[df['LABEL'].isin(['bp_sys', 'bp_dia', 'map'])]
    bp_pivot = bp_df.pivot_table(
        index=['SUBJECT_ID', 'CHARTTIME', 'HADM_ID'], 
        columns='LABEL', 
        values='VALUENUM'
    ).reset_index()

    for col in ['bp_sys', 'bp_dia', 'map']:
        if col not in bp_pivot.columns:
            bp_pivot[col] = np.nan

    missing_map_mask = bp_pivot['map'].isna() & bp_pivot['bp_sys'].notna() & bp_pivot['bp_dia'].notna()
    missing_maps = bp_pivot[missing_map_mask].copy()

    if not missing_maps.empty:
        missing_maps['VALUENUM'] = (missing_maps['bp_sys'] + 2 * missing_maps['bp_dia']) / 3.0
        missing_maps['LABEL'] = 'map_calculated' 
        missing_maps['UNIFIED_LABEL'] = 'map'
        new_maps_df = missing_maps[['SUBJECT_ID', 'HADM_ID', 'CHARTTIME', 'LABEL', 'UNIFIED_LABEL', 'VALUENUM']]
        df = pd.concat([df, new_maps_df], ignore_index=True)


    # FiO2 PRE-FILLING (Room Air = 21.0)
    if df_admissions is not None:
        df['CHARTTIME'] = pd.to_datetime(df['CHARTTIME'])
        df_admissions['intime'] = pd.to_datetime(df_admissions['intime'])
        
        df_admissions['intime_floored'] = df_admissions['intime'].dt.floor('H')
        
        admissions_sub = df_admissions[['HADM_ID', 'intime_floored']].drop_duplicates()
        fio2_df = df[df['UNIFIED_LABEL'] == 'fio2'].merge(admissions_sub, on='HADM_ID', how='inner')
        fio2_df['hour_bin'] = np.floor((fio2_df['CHARTTIME'] - fio2_df['intime_floored']).dt.total_seconds() / 3600.0)
        
        existing_pairs = fio2_df[['HADM_ID', 'hour_bin']].drop_duplicates()
        existing_pairs['exists'] = True
        
        subjects = admissions_sub['HADM_ID'].unique()
        hours = np.arange(28)
        grid = pd.MultiIndex.from_product([subjects, hours], names=['HADM_ID', 'hour_bin']).to_frame(index=False)
        grid = grid.merge(admissions_sub, on='HADM_ID', how='inner')
        
        missing_grid = grid.merge(existing_pairs, on=['HADM_ID', 'hour_bin'], how='left')
        missing_grid = missing_grid[missing_grid['exists'].isnull()].copy()
        
        if not missing_grid.empty:
            missing_grid['CHARTTIME'] = missing_grid['intime_floored'] + pd.to_timedelta(missing_grid['hour_bin'], unit='h')
            missing_grid['LABEL'] = 'fio2_imputed'
            missing_grid['UNIFIED_LABEL'] = 'fio2'
            missing_grid['VALUENUM'] = 21.0
            
            missing_grid['ITEMID'] = 190.0
            missing_grid['VALUEUOM'] = '%'
            
            if mapping_ids is not None:
                missing_grid = missing_grid.merge(mapping_ids[['HADM_ID', 'SUBJECT_ID']], on='HADM_ID', how='left')
            else:
                missing_grid['SUBJECT_ID'] = np.nan
                
            new_fio2_df = missing_grid[['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM', 'LABEL', 'UNIFIED_LABEL']]
            df = pd.concat([df, new_fio2_df], ignore_index=True)

        df = df.merge(admissions_sub, on='HADM_ID', how='left')

    df['CHARTTIME'] = df['CHARTTIME'].dt.floor('H')
    df = df.sort_values(by=['SUBJECT_ID', 'CHARTTIME']).reset_index(drop=True)
    
    for col in ['SUBJECT_ID', 'ITEMID', 'HADM_ID']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    return df

control_phys_clean = clean_and_standardize_vitals(control_phys, neonate_control, mapping_ids)
control_phys_clean

In [ ]:
# Cleans and standardizes neonatal physiological data (sepsis cohort)
sepsis_phys_clean = clean_and_standardize_vitals(sepsis_phys, neonate_sepsis, mapping_ids)
sepsis_phys_clean

In [ ]:
def verify_24h_continuous_data(df_vitals, max_allowed_gap_hours=4.0):
    """
    Verifies that patients have continuous data for core variables in the first 24 hours.
    
    Args:
        df_vitals: Long-format dataframe with [HADM_ID, CHARTTIME, UNIFIED_LABEL, VALUENUM]
        df_admissions: Dataframe with [HADM_ID, intime]
        max_allowed_gap_hours: Maximum acceptable gap between readings in hours.
    
    Returns:
        patient_summary: DataFrame with Pass/Fail status per HADM_ID.
        detailed_stats: DataFrame with gap and coverage stats per HADM_ID and Variable.
    """
    df = df_vitals.copy()
    df['CHARTTIME'] = pd.to_datetime(df['CHARTTIME'])

    if 'HADM_ID' in df_vitals.columns:
        df['HADM_ID'] = pd.to_numeric(df['HADM_ID'], errors='coerce').astype('Int64')
    
    # Merge using the floored intime and calculate relative hour offsets
    df['offset_hours'] = (df['CHARTTIME'] - df['intime_floored']).dt.total_seconds() / 3600.0
    
    # Filter to the core variables and the 0-24 hour window
    core_vars = ['heart_rate', 'sa02', 'fio2', 'respiratory_rate', 'temperature']
    df = df[df['UNIFIED_LABEL'].isin(core_vars)]
    df_24h = df[(df['offset_hours'] >= -1.0) & (df['offset_hours'] <= 24.0)].copy()
    
    # Sort to calculate accurate time gaps between consecutive readings
    df_24h = df_24h.sort_values(by=['HADM_ID', 'UNIFIED_LABEL', 'offset_hours'])
    
    # Calculate the time gap from the previous reading for the same patient and variable
    df_24h['time_gap'] = df_24h.groupby(['HADM_ID', 'UNIFIED_LABEL'])['offset_hours'].diff()
    
    # Aggregate stats per Patient per Variable
    detailed_stats = df_24h.groupby(['HADM_ID', 'UNIFIED_LABEL']).agg(
        first_record_hr=('offset_hours', 'min'),
        last_record_hr=('offset_hours', 'max'),
        max_gap_hr=('time_gap', 'max'),
        total_records=('VALUENUM', 'count')
    ).reset_index()
    
    # Handle NaNs in max_gap (occurs if there's only 1 record)
    detailed_stats['max_gap_hr'] = detailed_stats['max_gap_hr'].fillna(26.0)
    
    # Define the "Pass" Criteria for each variable
    detailed_stats['var_passed'] = (
        (detailed_stats['first_record_hr'] <= 2.0) &
        (detailed_stats['last_record_hr'] >= 22.0) &
        (detailed_stats['last_record_hr'] - detailed_stats['first_record_hr'] >= 22.0) &
        (detailed_stats['max_gap_hr'] <= max_allowed_gap_hours)
    )
    
    # Roll up to the Patient Level
    patient_summary = detailed_stats.groupby('HADM_ID').agg(
        vars_present=('UNIFIED_LABEL', 'nunique'),
        vars_passed=('var_passed', 'sum')
    ).reset_index()
    
    # Patient passes if they have all 5 variables AND all 5 variables passed the gap/coverage check
    patient_summary['has_24h_continuous'] = (
        (patient_summary['vars_present'] == len(core_vars)) & 
        (patient_summary['vars_passed'] == len(core_vars))
    )
    valid_hids = patient_summary[patient_summary['has_24h_continuous']]['HADM_ID']
    df_valid_24h = df_24h[(df_24h['HADM_ID'].isin(valid_hids)) & 
                      (df_24h['offset_hours'] >= -1.0) & 
                      (df_24h['offset_hours'] <= 24.0)].copy()
    
    valid_stats = detailed_stats[detailed_stats['HADM_ID'].isin(valid_hids)]

    total_patients = len(patient_summary)
    passing_patients = patient_summary['has_24h_continuous'].sum()
    
    print("\n" + "="*40)
    print(" 24-HOUR CONTINUOUS DATA SUMMARY ")
    print("="*40)
    print(f"Total patients in initial cohort: {total_patients:,}")
    print(f"Patients passing all criteria:    {passing_patients:,}")
    
    if total_patients > 0:
        retention_rate = (passing_patients / total_patients) * 100
        print(f"Retention rate:                   {retention_rate:.2f}%")
        
        # Breakdown of why patients failed (optional but highly helpful for debugging)
        failed_patients = total_patients - passing_patients
        if failed_patients > 0:
            missing_vars = (patient_summary['vars_present'] < len(core_vars)).sum()
            gap_failures = failed_patients - missing_vars
            print(f"\nFailure Breakdown:")
            print(f"  - Missing core variables:       {missing_vars:,} patients")
            print(f"  - Unacceptable time gaps/span:  {gap_failures:,} patients")
    print("="*40 + "\n")
    
    return df_valid_24h.drop(columns=['offset_hours', 'time_gap']), valid_stats

In [ ]:
# Verifies all physiological data is continuous for first 24 hours
control_phys_full, control_phys_stats = verify_24h_continuous_data(control_phys_clean, max_allowed_gap_hours=4.0)
sepsis_phys_full, sepsis_phys_stats = verify_24h_continuous_data(sepsis_phys_clean, max_allowed_gap_hours=4.0)

In [ ]:
# Constructs a dynamic source tensor from the physiological data

def build_dynamic_source_tensor(df_vitals, df_stats, cohort_label):
    core_vitals = ['heart_rate', 'temperature', 'respiratory_rate', 'fio2', 'sa02']

    source_labels = [
        'temp_c', 'temp_axillary_f', 'temp_rectal_f',
        'fio2', 'fio2_imputed',
        'heart_rate', 'respiratory_rate', 'sa02'
    ]

    channel_names = (
        core_vitals +
        [f'{v}_observed_mask' for v in core_vitals] +
        [f'source_{s}' for s in source_labels]
    )

    vital_to_idx = {v: i for i, v in enumerate(core_vitals)}
    source_to_idx = {s: i + (2 * len(core_vitals)) for i, s in enumerate(source_labels)}

    min_start_map = df_stats.groupby('HADM_ID')['first_record_hr'].min().to_dict()
    patient_ids = np.asarray(df_stats['HADM_ID'].unique(), dtype=np.int64)

    subject_lookup = (
        df_vitals[['HADM_ID', 'SUBJECT_ID']]
        .dropna()
        .drop_duplicates('HADM_ID')
        .assign(
            HADM_ID=lambda d: pd.to_numeric(d['HADM_ID']).astype('int64'),
            SUBJECT_ID=lambda d: pd.to_numeric(d['SUBJECT_ID']).astype('int64')
        )
        .set_index('HADM_ID')['SUBJECT_ID']
        .to_dict()
    )
    subject_ids = np.array([subject_lookup[int(hid)] for hid in patient_ids], dtype=np.int64)

    n_patients = len(patient_ids)
    n_channels = len(channel_names)

    X = np.full((n_patients, 24, n_channels), np.nan)
    y = np.full(n_patients, cohort_label)

    for i, hid in enumerate(patient_ids):
        p_data = df_vitals[df_vitals['HADM_ID'] == hid].copy()
        start_hr = min_start_map[hid]

        p_data['hour_bin'] = np.floor(
            (p_data['CHARTTIME'] - p_data['intime_floored']).dt.total_seconds() / 3600.0
        ).astype(int)

        p_data['tensor_idx'] = p_data['hour_bin'] - start_hr
        p_data = p_data[(p_data['tensor_idx'] >= 0) & (p_data['tensor_idx'] < 24)]

        grouped = p_data.groupby(['tensor_idx', 'UNIFIED_LABEL', 'LABEL'])

        for (t_idx, unified_lbl, raw_lbl), group in grouped:
            t_idx = int(t_idx)

            if unified_lbl in vital_to_idx:
                v_idx = vital_to_idx[unified_lbl]
                new_val = group['VALUENUM'].mean()

                if np.isnan(X[i, t_idx, v_idx]):
                    X[i, t_idx, v_idx] = new_val
                else:
                    X[i, t_idx, v_idx] = (X[i, t_idx, v_idx] + new_val) / 2.0

                X[i, t_idx, v_idx + len(core_vitals)] = 1.0

            if raw_lbl in source_to_idx:
                X[i, t_idx, source_to_idx[raw_lbl]] = 1.0

    X[:, :, len(core_vitals):] = np.nan_to_num(X[:, :, len(core_vitals):], nan=0.0)

    return X, y, patient_ids, subject_ids, channel_names

X_control, y_control, hadm_control, subject_control, channel_names = build_dynamic_source_tensor(
    control_phys_full, control_phys_stats, cohort_label=0
)

X_sepsis, y_sepsis, hadm_sepsis, subject_sepsis, _ = build_dynamic_source_tensor(
    sepsis_phys_full, sepsis_phys_stats, cohort_label=1
)

X_combined = np.concatenate([X_control, X_sepsis], axis=0)
y_combined = np.concatenate([y_control, y_sepsis], axis=0)
hadm_ids_combined = np.concatenate([hadm_control, hadm_sepsis], axis=0)
subject_ids_combined = np.concatenate([subject_control, subject_sepsis], axis=0)

In [ ]:
print(f"Tensor Shape: {X_combined.shape}")

value_nans = np.isnan(X_combined[:, :, 0:5]).sum()
print(f"Total NaNs in value channels: {value_nans}")

mask_nans = np.isnan(X_combined[:, :, 5:]).sum()
print(f"Total NaNs in mask/source channels: {mask_nans}")

unique_mask_vals = np.unique(X_combined[:, :, 5:])
print(f"Unique values in mask/source channels: {unique_mask_vals}")

first_step_masks = X_combined[:, 0, 5:10].sum(axis=1)
patients_with_empty_start = np.sum(first_step_masks == 0)
print(f"Patients with no observed core vital data at T=0: {patients_with_empty_start}")

In [ ]:
# source_temp_axillary_f is channel 11:
# 0-4 values, 5-9 observed masks, 10-17 source masks.
axillary_channel = channel_names.index('source_temp_axillary_f')
temp_value_channel = channel_names.index('temperature')
temp_mask_channel = channel_names.index('temperature_observed_mask')

axillary_indices = np.where(X_combined[:, :, axillary_channel] == 1)

if len(axillary_indices[0]) > 0:
    p_idx, t_idx = axillary_indices[0][0], axillary_indices[1][0]

    print(f"Sample Check - Patient {p_idx} at Hour {t_idx}:")
    print(f"  Axillary Source Mask: {X_combined[p_idx, t_idx, axillary_channel]}")
    print(f"  Temp Value:           {X_combined[p_idx, t_idx, temp_value_channel]}")
    print(f"  Temp Observed Mask:   {X_combined[p_idx, t_idx, temp_mask_channel]}")

In [ ]:
# Pick a random patient
sample_idx = 56 
patient_id = '199918'

# Plot Heart Rate (Channel 0) and its Mask (Channel 6)
plt.figure(figsize=(12, 4))
plt.step(range(24), X_combined[sample_idx, :, 0], where='post', label='Heart Rate (Value)')
plt.bar(range(24), X_combined[sample_idx, :, 6], alpha=0.3, label='Real Data Mask', color='orange')
plt.title(f"Tensor Check for HADM_ID: {patient_id}")
plt.xlabel("Tensor Index (Aligned to First Record)")
plt.legend()
plt.show()

In [ ]:
def interpolate_tensor_values(X_tensor, num_value_channels=5):
    """
    Applies linear interpolation to fill gaps in the value channels of a 3D tensor.
    Edges (Hour 0 or Hour 23) are filled using forward/backward fill.
    Mask channels are left completely untouched.
    """
    # Create a copy to prevent overwriting your original raw tensor in memory
    X_filled = np.copy(X_tensor)
    
    n_patients = X_filled.shape[0]
    
    for p in range(n_patients):
        for f in range(num_value_channels):
            # Extract the 24-hour sequence for this specific patient and feature
            series = pd.Series(X_filled[p, :, f])
            
            # interpolate(limit_direction='both') performs linear interpolation 
            # for internal gaps, AND handles forward/backward fill for the edges.
            series = series.interpolate(method='linear', limit_direction='both')
            
            # Fallback: If a patient somehow had 0 real readings for a feature 
            # (which your density filter should prevent), fill with 0 to avoid breaking the model.
            series = series.fillna(0)
            
            # Inject the filled data back into the tensor
            X_filled[p, :, f] = series.values
            
    return X_filled

# Execute on your combined tensor
X_interpolated = interpolate_tensor_values(X_combined, num_value_channels=5)

In [ ]:
# 1. Check Values (Should be 0 NaNs)
val_nans_before = np.isnan(X_combined[:, :, 0:5]).sum()
val_nans_after = np.isnan(X_interpolated[:, :, 0:5]).sum()

print(f"NaNs in Value Channels -> Before: {val_nans_before:,} | After: {val_nans_after}")

# 2. Check Masks (Should have no NaNs, and only contain 0.0 or 1.0)
unique_masks = np.unique(X_interpolated[:, :, 5:])
print(f"Unique values in Mask Channels: {unique_masks}")

In [ ]:
# Builds a train/val/test split and normalizes the data
from sklearn.model_selection import train_test_split

def split_and_normalize_dataset(X, y, hadm_ids, subject_ids, test_size=0.10, val_size=0.10, num_vitals=5):
    X_train_val, X_test, y_train_val, y_test, hadm_train_val, hadm_test, subj_train_val, subj_test = train_test_split(
        X, y, hadm_ids, subject_ids,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    adj_val_size = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val, hadm_train, hadm_val, subj_train, subj_val = train_test_split(
        X_train_val, y_train_val, hadm_train_val, subj_train_val,
        test_size=adj_val_size,
        random_state=42,
        stratify=y_train_val
    )

    def apply_normalization(tensor, stats):
        t_norm = np.copy(tensor)
        for f in range(num_vitals):
            t_norm[:, :, f] = (t_norm[:, :, f] - stats[f]['mean']) / stats[f]['std']
        return t_norm

    train_stats = {}
    for f in range(num_vitals):
        real_data = X_train[:, :, f][X_train[:, :, f + num_vitals] == 1.0]
        train_stats[f] = {
            'mean': np.mean(real_data),
            'std': np.std(real_data) if np.std(real_data) > 0 else 1e-8
        }

    X_train_norm = apply_normalization(X_train, train_stats)
    X_val_norm = apply_normalization(X_val, train_stats)
    X_test_norm = apply_normalization(X_test, train_stats)

    split_ids = {
        'train_hadm_ids': hadm_train,
        'val_hadm_ids': hadm_val,
        'test_hadm_ids': hadm_test,
        'train_subject_ids': subj_train,
        'val_subject_ids': subj_val,
        'test_subject_ids': subj_test,
    }

    return (X_train_norm, y_train), (X_val_norm, y_val), (X_test_norm, y_test), train_stats, split_ids

(X_train, y_train), (X_val, y_val), (X_test, y_test), final_stats, split_ids = split_and_normalize_dataset(
    X_interpolated,
    y_combined,
    hadm_ids_combined,
    subject_ids_combined,
    num_vitals=5
)

In [ ]:
# Exports the data to a .npz file
norm_params = np.array([[final_stats[i]['mean'], final_stats[i]['std']] for i in range(len(final_stats))])

np.savez_compressed(
    'neonatal_sepsis_data_v1.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    norm_params=norm_params,
    channel_names=np.array(channel_names),
    label_names=np.array(['control', 'sepsis']),
    **split_ids
)

In [ ]:
# Generate demographic breakdown for Table 1
control_ids = control_phys_full['SUBJECT_ID'].unique()
sepsis_ids = sepsis_phys_full['SUBJECT_ID'].unique()

control_patients = control_patients[control_patients['SUBJECT_ID'].isin(control_ids)]
control_admissions = control_admissions[control_admissions['SUBJECT_ID'].isin(control_ids)]

sepsis_patients = sepsis_patients[sepsis_patients['SUBJECT_ID'].isin(sepsis_ids)]
sepsis_admissions = sepsis_admissions[sepsis_admissions['SUBJECT_ID'].isin(sepsis_ids)]

print(control_ids.size)
print(sepsis_ids.size)

In [ ]:
gender_count = control_patients['GENDER'].value_counts(normalize=True)
gender_count

In [ ]:
gender_count = sepsis_patients['GENDER'].value_counts(normalize=True)
gender_count

In [ ]:
sepsis_demographics = sepsis_admissions['ETHNICITY'].value_counts()
sepsis_demographics

In [ ]:
control_demographics = control_admissions['ETHNICITY'].value_counts()
control_demographics

In [ ]:
def get_cohort_vitals_stats(df_cohort, cohort_name="Unnamed"):
    """
    Calculates Mean and Std Dev for a single cohort's physiological data.
    
    Args:
        df_cohort: The long-format dataframe (control_phys_full or sepsis_phys_full).
        cohort_name: String label for the printout.
    """
    # 1. Define variables of interest
    core_vars = ['heart_rate', 'temperature', 'respiratory_rate', 'fio2', 'sa02']
    
    # 2. Filter and calculate
    stats = (df_cohort[df_cohort['UNIFIED_LABEL'].isin(core_vars)]
             .groupby('UNIFIED_LABEL')['VALUENUM']
             .agg(['mean', 'std', 'min', 'max', 'count'])
             .reset_index())
    
    # Formatting for display
    stats.columns = ['Variable', 'Mean', 'StdDev', 'Min', 'Max', 'Count']
    
    print(f"\n" + "="*60)
    print(f" STATISTICAL SUMMARY: {cohort_name.upper()} COHORT ")
    print("="*60)
    print(stats.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print("="*60 + "\n")
    
    return stats

# Usage:
control_stats_summary = get_cohort_vitals_stats(control_phys_full, "Control")
sepsis_stats_summary = get_cohort_vitals_stats(sepsis_phys_full, "Sepsis")

In [ ]:
# Sets up collection of static features from MIMIC-III

MIMIC_ROOT = Path("/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4")
DYNAMIC_ARTIFACT_PATH = Path("neonatal_sepsis_data_v1.npz")
LATE_FUSION_ARTIFACT_PATH = Path("neonatal_sepsis_late_fusion_data_v1.npz")

STATIC_ITEMIDS = {
    "birth_weight_kg": [4183, 3723],
    "gestational_age_weeks": [3446],
    "apgar_1min": [4184],
}
STATIC_ITEMID_TO_FEATURE = {
    4183: "birth_weight_kg",
    3723: "birth_weight_kg",
    3446: "gestational_age_weeks",
    4184: "apgar_1min",
}
STATIC_FEATURE_BASE = ["birth_weight_kg", "gestational_age_weeks", "apgar_1min", "sex"]

if not DYNAMIC_ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Dynamic artifact not found: {DYNAMIC_ARTIFACT_PATH}")


def build_split_index(npz_obj):
    split_frames = []
    for split in ("train", "val", "test"):
        hadm_ids = np.asarray(npz_obj[f"{split}_hadm_ids"]).astype(np.int64)
        subject_ids = np.asarray(npz_obj[f"{split}_subject_ids"]).astype(np.int64)
        y = np.asarray(npz_obj[f"y_{split}"]).astype(np.int64)

        if not (len(hadm_ids) == len(subject_ids) == len(y)):
            raise ValueError(
                f"Split length mismatch for {split}: "
                f"hadm={len(hadm_ids)}, subject={len(subject_ids)}, y={len(y)}"
            )

        split_frames.append(
            pd.DataFrame(
                {
                    "split": split,
                    "row_idx": np.arange(len(hadm_ids), dtype=np.int64),
                    "hadm_id": hadm_ids,
                    "subject_id": subject_ids,
                    "y": y,
                }
            )
        )

    split_index = pd.concat(split_frames, ignore_index=True)

    duplicated_hadm = split_index["hadm_id"].duplicated().any()
    if duplicated_hadm:
        raise ValueError("Encountered duplicated HADM_IDs across splits; expected one row per admission.")

    return split_index


dynamic_npz = np.load(DYNAMIC_ARTIFACT_PATH, allow_pickle=True)
split_index_df = build_split_index(dynamic_npz)

cohort_hadm_ids = set(split_index_df["hadm_id"].astype(np.int64).tolist())
cohort_subject_ids = set(split_index_df["subject_id"].astype(np.int64).tolist())

print(f"Loaded dynamic artifact from: {DYNAMIC_ARTIFACT_PATH}")
print(f"Total cohort admissions: {len(cohort_hadm_ids):,}")
print(f"Total cohort subjects:   {len(cohort_subject_ids):,}")
print("Split sizes:")
print(split_index_df.groupby("split")["hadm_id"].count())
print("\nOutcome prevalence by split (mean y):")
print(split_index_df.groupby("split")["y"].mean().round(4))

split_index_df.head()

In [ ]:
# Helper methods for parsing static features

def parse_numeric_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text == "":
        return np.nan

    # Keep only the first numeric token.
    m = re.search(r"[-+]?\d*\.?\d+", text)
    if not m:
        return np.nan
    try:
        return float(m.group(0))
    except ValueError:
        return np.nan


def parse_birth_weight_kg(row):
    raw_num = pd.to_numeric(row.get("VALUENUM"), errors="coerce")
    raw_txt = row.get("VALUE")
    val = raw_num if pd.notna(raw_num) else parse_numeric_text(raw_txt)

    if pd.isna(val):
        return np.nan, "missing_or_unparseable"

    # Already in plausible kg range.
    if 0.3 <= val <= 7.0:
        return float(val), "ok_kg"

    # Likely documented in grams, convert to kg.
    if 300 <= val <= 7000:
        return float(val / 1000.0), "converted_g_to_kg"

    return np.nan, "out_of_range"


def parse_gestational_age_weeks(row):
    raw_num = pd.to_numeric(row.get("VALUENUM"), errors="coerce")
    raw_txt = row.get("VALUE")

    if pd.notna(raw_num) and 20 <= raw_num <= 45:
        return float(raw_num), "ok_numeric"

    text = "" if pd.isna(raw_txt) else str(raw_txt).lower().strip()
    if text == "":
        return np.nan, "missing_or_unparseable"

    m_frac = re.search(r"(\d{1,2})\s+(\d)\s*/\s*7", text)
    if m_frac:
        weeks = int(m_frac.group(1))
        days = int(m_frac.group(2))
        val = weeks + (days / 7.0)
        if 20 <= val <= 45 and 0 <= days <= 6:
            return float(val), "parsed_text_fraction"

    m_plus = re.search(r"(\d{1,2})\s*\+\s*(\d)", text)
    if m_plus:
        weeks = int(m_plus.group(1))
        days = int(m_plus.group(2))
        val = weeks + (days / 7.0)
        if 20 <= val <= 45 and 0 <= days <= 6:
            return float(val), "parsed_text_plus"

    m_wd = re.search(r"(\d{1,2})\s*w(?:eeks?)?\s*(\d)?\s*d?", text)
    if m_wd:
        weeks = int(m_wd.group(1))
        days = int(m_wd.group(2)) if m_wd.group(2) else 0
        val = weeks + (days / 7.0)
        if 20 <= val <= 45 and 0 <= days <= 6:
            return float(val), "parsed_text_wd"

    fallback = parse_numeric_text(text)
    if pd.notna(fallback) and 20 <= fallback <= 45:
        return float(fallback), "parsed_text_numeric"

    return np.nan, "out_of_range_or_unparseable"


def parse_apgar_1min(row):
    raw_num = pd.to_numeric(row.get("VALUENUM"), errors="coerce")
    raw_txt = row.get("VALUE")

    val = raw_num if pd.notna(raw_num) else parse_numeric_text(raw_txt)
    if pd.isna(val):
        return np.nan, "missing_or_unparseable"

    if 0 <= val <= 10:
        return float(val), "ok"

    return np.nan, "out_of_range"


def parse_static_row(row):
    var = row.get("variable")

    if var == "birth_weight_kg":
        return parse_birth_weight_kg(row)
    if var == "gestational_age_weeks":
        return parse_gestational_age_weeks(row)
    if var == "apgar_1min":
        return parse_apgar_1min(row)

    return np.nan, "unknown_variable"

In [ ]:
# Extracts static features from chartevents data
needed_itemids = set()
for item_list in STATIC_ITEMIDS.values():
    needed_itemids.update(item_list)

admissions_map = pd.read_csv(
    MIMIC_ROOT / "ADMISSIONS.csv",
    usecols=["SUBJECT_ID", "HADM_ID", "ADMITTIME", "DISCHTIME"],
)
admissions_map["SUBJECT_ID"] = pd.to_numeric(admissions_map["SUBJECT_ID"], errors="coerce").astype("Int64")
admissions_map["HADM_ID"] = pd.to_numeric(admissions_map["HADM_ID"], errors="coerce").astype("Int64")
admissions_map["ADMITTIME"] = pd.to_datetime(admissions_map["ADMITTIME"], errors="coerce")
admissions_map["DISCHTIME"] = pd.to_datetime(admissions_map["DISCHTIME"], errors="coerce")

admissions_map = admissions_map[
    admissions_map["SUBJECT_ID"].isin(cohort_subject_ids) | admissions_map["HADM_ID"].isin(cohort_hadm_ids)
].copy()

subject_windows = {}
for sid, g in admissions_map.dropna(subset=["SUBJECT_ID", "HADM_ID"]).groupby("SUBJECT_ID"):
    rows = []
    for _, r in g.iterrows():
        if pd.notna(r["ADMITTIME"]):
            start = r["ADMITTIME"] - pd.Timedelta(days=1)
            if pd.notna(r["DISCHTIME"]):
                end = r["DISCHTIME"] + pd.Timedelta(days=1)
            else:
                end = r["ADMITTIME"] + pd.Timedelta(days=7)
        else:
            start = pd.Timestamp.min
            end = pd.Timestamp.max
        rows.append((start, end, int(r["HADM_ID"])))
    subject_windows[int(sid)] = rows


def infer_hadm_from_subject_charttime(subject_id, charttime):
    if pd.isna(subject_id) or pd.isna(charttime):
        return np.nan

    windows = subject_windows.get(int(subject_id), [])
    if not windows:
        return np.nan

    matches = [hadm for start, end, hadm in windows if start <= charttime <= end]
    if len(matches) == 1:
        return matches[0]
    return np.nan


chartevents_cols = ["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM", "VALUE", "VALUEUOM"]
chartevents_path = MIMIC_ROOT / "CHARTEVENTS.csv"
chunk_size = 5_000_000

static_event_chunks = []
rows_seen = 0
rows_after_item_filter = 0
rows_after_cohort_filter = 0
rows_hadm_inferred = 0

for i, chunk in enumerate(pd.read_csv(chartevents_path, usecols=chartevents_cols, chunksize=chunk_size)):
    rows_seen += len(chunk)

    chunk = chunk[chunk["ITEMID"].isin(needed_itemids)].copy()
    if chunk.empty:
        continue
    rows_after_item_filter += len(chunk)

    chunk["SUBJECT_ID"] = pd.to_numeric(chunk["SUBJECT_ID"], errors="coerce").astype("Int64")
    chunk["HADM_ID"] = pd.to_numeric(chunk["HADM_ID"], errors="coerce")
    chunk["CHARTTIME"] = pd.to_datetime(chunk["CHARTTIME"], errors="coerce")

    chunk = chunk[
        chunk["HADM_ID"].isin(cohort_hadm_ids) | chunk["SUBJECT_ID"].isin(cohort_subject_ids)
    ].copy()
    if chunk.empty:
        continue

    missing_hadm_mask = chunk["HADM_ID"].isna() & chunk["SUBJECT_ID"].notna() & chunk["CHARTTIME"].notna()
    if missing_hadm_mask.any():
        inferred = chunk.loc[missing_hadm_mask, ["SUBJECT_ID", "CHARTTIME"]].apply(
            lambda r: infer_hadm_from_subject_charttime(r["SUBJECT_ID"], r["CHARTTIME"]),
            axis=1,
        )
        rows_hadm_inferred += int(pd.notna(inferred).sum())
        chunk.loc[missing_hadm_mask, "HADM_ID"] = inferred.values

    chunk = chunk[chunk["HADM_ID"].isin(cohort_hadm_ids)].copy()
    if chunk.empty:
        continue
    rows_after_cohort_filter += len(chunk)

    chunk["HADM_ID"] = chunk["HADM_ID"].astype(np.int64)
    chunk["variable"] = chunk["ITEMID"].map(STATIC_ITEMID_TO_FEATURE)

    parsed = chunk.apply(parse_static_row, axis=1, result_type="expand")
    parsed.columns = ["parsed_value", "parse_status"]
    chunk = pd.concat([chunk, parsed], axis=1)

    static_event_chunks.append(
        chunk[["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM", "VALUE", "VALUEUOM", "variable", "parsed_value", "parse_status"]]
    )

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} chunks ...")

if not static_event_chunks:
    raise RuntimeError("No static chart events found for the selected cohort and item IDs.")

static_events_df = pd.concat(static_event_chunks, ignore_index=True)

valid_events_df = static_events_df[static_events_df["parsed_value"].notna()].copy()
valid_events_df = valid_events_df.sort_values(["HADM_ID", "variable", "CHARTTIME"], kind="mergesort")

first_valid_df = (
    valid_events_df
    .groupby(["HADM_ID", "variable"], as_index=False)
    .first()[["HADM_ID", "variable", "parsed_value", "ITEMID", "CHARTTIME", "parse_status"]]
)

print(f"Rows scanned from CHARTEVENTS:      {rows_seen:,}")
print(f"Rows after ITEMID filter:           {rows_after_item_filter:,}")
print(f"Rows after cohort+HADM filtering:   {rows_after_cohort_filter:,}")
print(f"Rows with HADM inferred by timing:  {rows_hadm_inferred:,}")
print(f"Total extracted static rows:        {len(static_events_df):,}")
print(f"Rows with valid parsed values:      {len(valid_events_df):,}")

first_valid_df.head()

In [ ]:
# Merges static features with demographic data

static_wide_df = first_valid_df.pivot(index="HADM_ID", columns="variable", values="parsed_value").reset_index()

base_static_df = split_index_df[["hadm_id", "subject_id", "split", "y"]].drop_duplicates(subset=["hadm_id"]).copy()
static_raw_df = base_static_df.merge(
    static_wide_df,
    left_on="hadm_id",
    right_on="HADM_ID",
    how="left",
).drop(columns=["HADM_ID"], errors="ignore")

patients_df = pd.read_csv(MIMIC_ROOT / "PATIENTS.csv", usecols=["SUBJECT_ID", "GENDER"])
patients_df["SUBJECT_ID"] = pd.to_numeric(patients_df["SUBJECT_ID"], errors="coerce").astype("Int64")
patients_df = patients_df.dropna(subset=["SUBJECT_ID"]).drop_duplicates(subset=["SUBJECT_ID"])

static_raw_df = static_raw_df.merge(
    patients_df.rename(columns={"SUBJECT_ID": "subject_id", "GENDER": "sex"}),
    on="subject_id",
    how="left",
)

static_raw_df["sex"] = (
    static_raw_df["sex"]
    .astype("string")
    .str.upper()
    .where(lambda s: s.isin(["M", "F"]))
)

for col in ["birth_weight_kg", "gestational_age_weeks", "apgar_1min"]:
    static_raw_df[col] = pd.to_numeric(static_raw_df[col], errors="coerce")

parse_status_counts_df = (
    static_events_df
    .groupby(["variable", "parse_status"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["variable", "n_rows"], ascending=[True, False])
)

conflict_df = (
    valid_events_df
    .groupby(["HADM_ID", "variable"], dropna=False)
    .agg(
        n_valid=("parsed_value", "size"),
        n_unique=("parsed_value", lambda s: int(pd.Series(np.round(s.astype(float), 4)).nunique())),
        min_value=("parsed_value", "min"),
        max_value=("parsed_value", "max"),
        first_charttime=("CHARTTIME", "min"),
        last_charttime=("CHARTTIME", "max"),
    )
    .reset_index()
)
conflict_df["has_conflict"] = conflict_df["n_unique"] > 1

conflict_summary_df = (
    conflict_df
    .groupby("variable", dropna=False)
    .agg(
        n_hadm_with_valid=("HADM_ID", "nunique"),
        n_hadm_with_conflict=("has_conflict", "sum"),
        pct_hadm_with_conflict=("has_conflict", "mean"),
        median_n_valid_entries=("n_valid", "median"),
    )
    .reset_index()
)

raw_coverage_df = (
    static_raw_df
    .groupby("split")[["birth_weight_kg", "gestational_age_weeks", "apgar_1min", "sex"]]
    .apply(lambda g: g.notna().mean())
    .reset_index()
)

missing_rows = []
for feature in ["birth_weight_kg", "gestational_age_weeks", "apgar_1min", "sex"]:
    tmp = (
        static_raw_df
        .assign(is_missing=static_raw_df[feature].isna().astype(float))
        .groupby(["split", "y"], dropna=False)["is_missing"]
        .agg(["mean", "sum", "count"])
        .reset_index()
        .rename(columns={"mean": "missing_rate", "sum": "n_missing", "count": "n_total"})
    )
    tmp["feature"] = feature
    missing_rows.append(tmp)
missing_by_split_outcome_df = pd.concat(missing_rows, ignore_index=True)

print("Raw static feature coverage by split (fraction observed):")
print(raw_coverage_df.round(4))
print("\nParser status counts:")
print(parse_status_counts_df)
print("\nConflict summary:")
print(conflict_summary_df.round(4))
print("\nMissingness by split and outcome (first 12 rows):")
print(missing_by_split_outcome_df.head(12).round(4))

static_raw_df.head()

In [ ]:
# Builds a train/val/test split and normalizes the data

continuous_base_cols = ["birth_weight_kg", "gestational_age_weeks", "apgar_1min"]

static_raw_by_hadm = static_raw_df.set_index("hadm_id")
split_static_tables = {}

for split in ("train", "val", "test"):
    hadm_ids = np.asarray(dynamic_npz[f"{split}_hadm_ids"]).astype(np.int64)
    subject_ids = np.asarray(dynamic_npz[f"{split}_subject_ids"]).astype(np.int64)
    y = np.asarray(dynamic_npz[f"y_{split}"]).astype(np.int64)

    split_table = static_raw_by_hadm.reindex(hadm_ids).reset_index()

    if not np.array_equal(split_table["hadm_id"].to_numpy(dtype=np.int64), hadm_ids):
        raise AssertionError(f"HADM_ID alignment mismatch for split={split}")

    if split_table["subject_id"].isna().any():
        raise AssertionError(f"Found missing subject_id after alignment for split={split}")

    aligned_subjects = split_table["subject_id"].to_numpy(dtype=np.int64)
    if not np.array_equal(aligned_subjects, subject_ids):
        raise AssertionError(f"SUBJECT_ID alignment mismatch for split={split}")

    aligned_y = split_table["y"].to_numpy(dtype=np.int64)
    if not np.array_equal(aligned_y, y):
        raise AssertionError(f"y alignment mismatch for split={split}")

    split_static_tables[split] = split_table

# Train-only summary stats on observed values.
train_table = split_static_tables["train"]
impute_values = {}
norm_params = {}
excluded_continuous_cols = []

for col in continuous_base_cols:
    observed = train_table[col].dropna().astype(float)
    if observed.empty:
        warnings.warn(
            f"No observed training values for {col}; excluding {col}_z and retaining missingness indicator only.",
            RuntimeWarning,
        )
        excluded_continuous_cols.append(col)
        continue

    impute_values[col] = float(observed.median())
    mean_val = float(observed.mean())
    std_val = float(observed.std(ddof=0))
    if std_val == 0:
        std_val = 1e-8
    norm_params[col] = (mean_val, std_val)

processed_split_tables = {}
X_static = {}

continuous_z_cols = [f"{col}_z" for col in continuous_base_cols if col in norm_params]
missing_indicator_cols = [f"{col}_missing" for col in continuous_base_cols] + ["sex_missing"]
categorical_cols = ["sex_F", "sex_M"]

feature_cols = continuous_z_cols + categorical_cols + missing_indicator_cols

for split in ("train", "val", "test"):
    df = split_static_tables[split].copy()
    proc = df[["hadm_id", "subject_id", "y", "split"]].copy()

    # Missingness indicators first, before imputation.
    for col in continuous_base_cols:
        proc[f"{col}_missing"] = df[col].isna().astype(np.float32)

    # Continuous impute + standardize using train-only parameters.
    for col in continuous_base_cols:
        if col not in norm_params:
            continue
        fill_val = impute_values[col]
        mean_val, std_val = norm_params[col]
        filled = df[col].astype(float).fillna(fill_val)
        proc[f"{col}_z"] = ((filled - mean_val) / std_val).astype(np.float32)

    # Sex encoding with explicit missingness signal.
    sex = df["sex"].astype("string").str.upper()
    proc["sex_F"] = (sex == "F").astype(np.float32)
    proc["sex_M"] = (sex == "M").astype(np.float32)
    proc["sex_missing"] = (~sex.isin(["F", "M"])).astype(np.float32)

    # Tensor assembly.
    X_static_split = proc[feature_cols].to_numpy(dtype=np.float32)
    if np.isnan(X_static_split).any() or np.isinf(X_static_split).any():
        raise AssertionError(f"Found NaN/Inf in X_static_{split}")

    X_static[split] = X_static_split
    processed_split_tables[split] = proc

for split in ("train", "val", "test"):
    if X_static[split].shape[0] != len(dynamic_npz[f"y_{split}"]):
        raise AssertionError(f"Row count mismatch in X_static_{split}")

processed_static_df = pd.concat([processed_split_tables["train"], processed_split_tables["val"], processed_split_tables["test"]], ignore_index=True)

static_impute_feature_names = np.array(continuous_base_cols, dtype=object)
static_impute_values = np.array([impute_values.get(col, np.nan) for col in continuous_base_cols], dtype=np.float32)
static_norm_feature_names = np.array(continuous_base_cols, dtype=object)
static_norm_params = np.array([
    [norm_params.get(col, (np.nan, np.nan))[0], norm_params.get(col, (np.nan, np.nan))[1]]
    for col in continuous_base_cols
], dtype=np.float32)

print("Static tensor features:", feature_cols)
print("Excluded continuous features (if any):", excluded_continuous_cols)
print("X_static_train shape:", X_static["train"].shape)
print("X_static_val shape:  ", X_static["val"].shape)
print("X_static_test shape: ", X_static["test"].shape)

In [ ]:
# Save static feature data

raw_csv_path = Path("neonatal_static_features_raw.csv")
processed_csv_path = Path("neonatal_static_features_processed.csv")
audit_csv_path = Path("neonatal_static_feature_audit.csv")

static_raw_df.to_csv(raw_csv_path, index=False)
processed_static_df.to_csv(processed_csv_path, index=False)

coverage_long_df = raw_coverage_df.melt(
    id_vars=["split"],
    value_vars=["birth_weight_kg", "gestational_age_weeks", "apgar_1min", "sex"],
    var_name="feature",
    value_name="observed_fraction",
)
coverage_long_df["audit_section"] = "coverage_by_split"

parse_status_export_df = parse_status_counts_df.copy()
parse_status_export_df["audit_section"] = "parse_status_counts"

conflict_export_df = conflict_summary_df.copy()
conflict_export_df["audit_section"] = "conflict_summary"

missing_export_df = missing_by_split_outcome_df.copy()
missing_export_df["audit_section"] = "missingness_by_split_outcome"

static_audit_df = pd.concat(
    [coverage_long_df, parse_status_export_df, conflict_export_df, missing_export_df],
    ignore_index=True,
    sort=False,
)
static_audit_df.to_csv(audit_csv_path, index=False)

np.savez_compressed(
    LATE_FUSION_ARTIFACT_PATH,
    X_train=dynamic_npz["X_train"],
    y_train=dynamic_npz["y_train"],
    X_val=dynamic_npz["X_val"],
    y_val=dynamic_npz["y_val"],
    X_test=dynamic_npz["X_test"],
    y_test=dynamic_npz["y_test"],
    X_static_train=X_static["train"],
    X_static_val=X_static["val"],
    X_static_test=X_static["test"],
    static_feature_names=np.array(feature_cols, dtype=object),
    static_raw_feature_names=np.array(STATIC_FEATURE_BASE, dtype=object),
    static_impute_feature_names=static_impute_feature_names,
    static_impute_values=static_impute_values,
    static_norm_feature_names=static_norm_feature_names,
    static_norm_params=static_norm_params,
    excluded_continuous_cols=np.array(excluded_continuous_cols, dtype=object),
    channel_names=dynamic_npz["channel_names"],
    label_names=dynamic_npz["label_names"],
    train_hadm_ids=dynamic_npz["train_hadm_ids"],
    val_hadm_ids=dynamic_npz["val_hadm_ids"],
    test_hadm_ids=dynamic_npz["test_hadm_ids"],
    train_subject_ids=dynamic_npz["train_subject_ids"],
    val_subject_ids=dynamic_npz["val_subject_ids"],
    test_subject_ids=dynamic_npz["test_subject_ids"],
)

print(f"Saved late-fusion artifact: {LATE_FUSION_ARTIFACT_PATH}")
print(f"Saved raw static table:     {raw_csv_path}")
print(f"Saved processed table:      {processed_csv_path}")
print(f"Saved static audit table:   {audit_csv_path}")

In [ ]:
# Check on static feature data

late_fusion_npz = np.load(LATE_FUSION_ARTIFACT_PATH, allow_pickle=True)

for split in ("train", "val", "test"):
    x_dyn = late_fusion_npz[f"X_{split}"]
    y = late_fusion_npz[f"y_{split}"]
    x_static = late_fusion_npz[f"X_static_{split}"]

    assert x_static.shape[0] == y.shape[0], f"X_static_{split} row mismatch"
    assert late_fusion_npz[f"{split}_hadm_ids"].shape[0] == y.shape[0], f"{split}_hadm_ids length mismatch"
    assert late_fusion_npz[f"{split}_subject_ids"].shape[0] == y.shape[0], f"{split}_subject_ids length mismatch"
    assert np.isfinite(x_static).all(), f"X_static_{split} contains non-finite values"

    print(
        f"{split}: X_dynamic={x_dyn.shape}, X_static={x_static.shape}, y={y.shape}, "
        f"positive_rate={y.mean():.4f}"
    )

print("Static feature names:", late_fusion_npz["static_feature_names"].tolist())
print("Static norm feature names:", late_fusion_npz["static_norm_feature_names"].tolist())
print("Static norm params:\n", late_fusion_npz["static_norm_params"])